# Kafka Producer/Consumer Test for LLM Service

This notebook allows you to produce and consume messages to test the LLM service.

In [ ]:
# Install required packages
#!pip install kafka-python

In [ ]:
import json
import time
from datetime import datetime
from kafka import KafkaProducer, KafkaConsumer
from kafka.errors import KafkaError

## Producer - Send Messages to agent-requests Topic

In [ ]:
# Initialize Kafka producer
producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

print("Kafka producer initialized successfully!")

In [ ]:
# Sample message for testing
test_message = {
    "header": {
        "id": "test-123",
        "timestamp": datetime.now().isoformat()
    },
    "payload": {
        "agent": {
            "context": {
                "query": "result for 2+2",
                "history":""
            },
            "prompt": "Answer the query: $query"
        }
    }
}

print("Sample message:")
print(json.dumps(test_message, indent=2))

In [ ]:
test_message['payload']['agent']['context']['query']="I want to buy the latest iPhone available in your cataluoge 15 pro red calor"
test_message['payload']['agent']['prompt']="""You are an expert Customer Service Agent. Analyze the user's query to understand their intent and plan the appropriate response.

1. REASONING: Understand what the user wants and determine what tools are needed
2. TOOL SELECTION: Decide which functions to call (search_products, search_faqs, or both)
3. PARAMETER EXTRACTION: Extract search parameters and filters from the query

AVAILABLE TOOLS:
- search_products: For finding products, recommendations, product details in Milvus database
- search_faqs: For questions about the business, shipping, returns, general info

OUTPUT JSON SCHEMA:
{{
    "reasoning": "Explanation of user intent and why specific tools are needed",
    "FunctionCall": [
        {{
            "name": "search_products",
            "args": {{
                "text": "combined search text with image descriptions",
                "filters": {{
                    "category": "string or None",
                    "price_range": {{
                        "min": number,
                        "max": number,
                        "operation": "eq" | "lt" | "gt" | "between"
                    }},
                    "attributes": {{
                        "color": "string or None",
                        "size": "string or None",
                        "brand": "string or None",
                        "material": "string or None"
                    }}
                }}
            }}
        }}
    ]
}}

REASONING GUIDELINES:
- If user asks about policies, shipping, returns (use search_faqs)
- If user wants to find products, recommendations (use search_products)
- If user needs both product info AND policy info use both tools
- Explain your reasoning clearly

TEXT OPTIMIZATION RULES:
- Combine user query keywords with image descriptions
- Remove conversational words, keep only searchable product attributes: colors, materials, styles, functions
- For images: describe style, color, material, shape, function, category

FILTER EXTRACTION RULES:
- category: Extract product category from query/image (e.g., "Desks / Office Desks", "Clothing / Dresses")
- price_range: Extract budget mentions (e.g., "under $100" → max: 100, operation: "lt")
- attributes: Extract specific product features (color, size, brand, material)

IMAGE ANALYSIS REQUIREMENTS:
- Generate dense, factual descriptions focusing on searchable attributes
- Include: style, color, material, shape, size indicators, function, category
- Example: "minimalist white desk with rectangular top and thin metal legs"
- DO NOT make up information about products you don't recognize

PRICE OPERATIONS:
- "eq": exact price match
- "lt": less than
- "gt": greater than
- "between": range between min and max

User Query: $query
History: $history
"""

In [ ]:
test_message

In [ ]:
test_message2={
  "header": {
    "id": "test-123",
    "timestamp": "2025-07-14T11:08:01.429668"
  },
  "payload": {
    "agent": {
      "context": {
        "query": "I want to buy the latest iPhone available in your cataluoge 15 pro red calor",
        "history": [
          {
            "interaction_id": "5677124b-c762-47b7-81a1-08580eb3d0b1",
            "timestamp": 1752476886.7546618,
            "user_query": "I want to buy the latest iPhone available in your cataluoge 15 pro red calor",
            "llm_reasoning": "The user is explicitly asking to find a specific product, 'the latest iPhone 15 Pro in red color'. This requires using the 'search_products' tool to query the product catalog.",
            "function_executions": [
              {
                "execution_id": "aa79fd0f-bcb2-4fdc-9301-4d10e5115232",
                "function_name": "search_products",
                "parameters": {
                  "text": "iPhone 15 Pro",
                  "filters": {
                    "category": None,
                    "price_range": None,
                    "attributes": {
                      "color": "red",
                      "size": None,
                      "brand": "Apple",
                      "material": None
                    }
                  }
                },
                "execution_status": "completed",
                "started_at": 1752476886.7546723,
                "completed_at": 1752476886.7786772,
                "execution_result": {
                  "products": [
                    {
                      "id": "iphone-15-pro-red",
                      "name": "iPhone 15 Pro",
                      "color": "Red",
                      "price": 999.99,
                      "category": "Electronics/Smartphones",
                      "brand": "Apple",
                      "in_stock": True,
                      "description": "Latest iPhone 15 Pro in stunning red color"
                    }
                  ],
                  "total_found": 1,
                  "search_text": "iPhone 15 Pro",
                  "filters_applied": {
                    "category": None,
                    "price_range": None,
                    "attributes": {
                      "color": "red",
                      "size": None,
                      "brand": "Apple",
                      "material": None
                    }
                  },
                  "timestamp": 1752476886.7697945
                },
                "error_details": None
              }
            ],
            "response_type": "function_assisted",
            "execution_summary": {
              "total_functions_executed": 1,
              "all_successful": True,
              "execution_results": [
                {
                  "products": [
                    {
                      "id": "iphone-15-pro-red",
                      "name": "iPhone 15 Pro",
                      "color": "Red",
                      "price": 999.99,
                      "category": "Electronics/Smartphones",
                      "brand": "Apple",
                      "in_stock": True,
                      "description": "Latest iPhone 15 Pro in stunning red color"
                    }
                  ],
                  "total_found": 1,
                  "search_text": "iPhone 15 Pro",
                  "filters_applied": {
                    "category": None,
                    "price_range": None,
                    "attributes": {
                      "color": "red",
                      "size": None,
                      "brand": "Apple",
                      "material": None
                    }
                  },
                  "timestamp": 1752476886.7697945
                }
              ]
            },
            "response": "I can definitely help you with that! You're looking for the latest iPhone, specifically the iPhone 15 Pro in red.\n\nHowever, it looks like the function execution results, which would tell me if the iPhone 15 Pro is indeed the latest model we have, if it's available in red, and its current price and stock, are missing.\n\nCould you please provide the function execution results? Once I have that information, I can give you a comprehensive answer about its availability and guide you through the next steps to purchase it!"
          }
        ]
      },
      "prompt": "You are an expert Customer Service Agent. Analyze the user's query to understand their intent and plan the appropriate response.\n\n1. REASONING: Understand what the user wants and determine what tools are needed\n2. TOOL SELECTION: Decide which functions to call (search_products, search_faqs, or both)\n3. PARAMETER EXTRACTION: Extract search parameters and filters from the query\n\nAVAILABLE TOOLS:\n- search_products: For finding products, recommendations, product details in Milvus database\n- search_faqs: For questions about the business, shipping, returns, general info\n\nOUTPUT JSON SCHEMA:\n{{\n    \"reasoning\": \"Explanation of user intent and why specific tools are needed\",\n    \"FunctionCall\": [\n        {{\n            \"name\": \"search_products\",\n            \"args\": {{\n                \"text\": \"combined search text with image descriptions\",\n                \"filters\": {{\n                    \"category\": \"string or None\",\n                    \"price_range\": {{\n                        \"min\": number,\n                        \"max\": number,\n                        \"operation\": \"eq\" | \"lt\" | \"gt\" | \"between\"\n                    }},\n                    \"attributes\": {{\n                        \"color\": \"string or None\",\n                        \"size\": \"string or None\",\n                        \"brand\": \"string or None\",\n                        \"material\": \"string or None\"\n                    }}\n                }}\n            }}\n        }}\n    ]\n}}\n\nREASONING GUIDELINES:\n- If user asks about policies, shipping, returns (use search_faqs)\n- If user wants to find products, recommendations (use search_products)\n- If user needs both product info AND policy info use both tools\n- Explain your reasoning clearly\n\nTEXT OPTIMIZATION RULES:\n- Combine user query keywords with image descriptions\n- Remove conversational words, keep only searchable product attributes: colors, materials, styles, functions\n- For images: describe style, color, material, shape, function, category\n\nFILTER EXTRACTION RULES:\n- category: Extract product category from query/image (e.g., \"Desks / Office Desks\", \"Clothing / Dresses\")\n- price_range: Extract budget mentions (e.g., \"under $100\" \u2192 max: 100, operation: \"lt\")\n- attributes: Extract specific product features (color, size, brand, material)\n\nIMAGE ANALYSIS REQUIREMENTS:\n- Generate dense, factual descriptions focusing on searchable attributes\n- Include: style, color, material, shape, size indicators, function, category\n- Example: \"minimalist white desk with rectangular top and thin metal legs\"\n- DO NOT make up information about products you don't recognize\n\nPRICE OPERATIONS:\n- \"eq\": exact price match\n- \"lt\": less than\n- \"gt\": greater than\n- \"between\": range between min and max\n\nUser Query: $query\nHistory: $history\n",
      "response": "I can definitely help you with that! You're looking for the latest iPhone, specifically the iPhone 15 Pro in red.\n\nHowever, it looks like the function execution results, which would tell me if the iPhone 15 Pro is indeed the latest model we have, if it's available in red, and its current price and stock, are missing.\n\nCould you please provide the function execution results? Once I have that information, I can give you a comprehensive answer about its availability and guide you through the next steps to purchase it!"
    }
  }
}
test_message2['payload']['agent']['context']['query']="how much it cost and how many days of shipping"

In [ ]:
# Send message to agent-requests topic
try:
    future = producer.send('agent-requests', test_message2)
    record_metadata = future.get(timeout=10)
    print(f"Message sent successfully!")
    print(f"Topic: {record_metadata.topic}")
    print(f"Partition: {record_metadata.partition}")
    print(f"Offset: {record_metadata.offset}")
except KafkaError as e:
    print(f"Error sending message: {e}")
finally:
    producer.flush()

## Consumer - Listen for Responses

In [ ]:
# Initialize Kafka consumer for agent-responses topic
consumer = KafkaConsumer(
    'agent-responses',
    #'search_products',
    bootstrap_servers=['localhost:9092'],
    value_deserializer=lambda m: json.loads(m.decode('utf-8')),
    auto_offset_reset='earliest',  # Changed from 'latest' to 'earliest'
    consumer_timeout_ms=10000  # 10 seconds timeout
)

print("Kafka consumer initialized for agent-responses topic!")
print("Listening for messages (10 second timeout)...")

In [ ]:
# Listen for messages
try:
    for message in consumer:
        print(f"\nReceived message:")
        print(f"Topic: {message.topic}")
        print(f"Partition: {message.partition}")
        print(f"Offset: {message.offset}")
        print(f"Value: {json.dumps(message.value, indent=2)}")
        
        # Check if this message has the response field
        response_field = message.value.get("payload", {}).get("agent", {}).get("response")
        if response_field:
            print(f"\n🎯 RESPONSE FIELD FOUND: {response_field}")
 
except Exception as e:
    print(f"No messages received or error: {e}")
finally:
    consumer.close()

In [ ]:
1/0

## Custom Message Sender

In [ ]:
def send_custom_message(query, prompt_template="Answer the query: $query"):
    """Send a custom message to the LLM service"""
    
    message = {
        "header": {
            "id": f"custom-{int(time.time())}",
            "timestamp": datetime.now().isoformat()
        },
        "payload": {
            "agent": {
                "context": {
                    "query": query
                },
                "prompt": prompt_template
            }
        }
    }
    
    try:
        future = producer.send('agent-requests', message)
        record_metadata = future.get(timeout=10)
        print(f"Custom message sent successfully!")
        print(f"Query: {query}")
        print(f"Offset: {record_metadata.offset}")
        return True
    except KafkaError as e:
        print(f"Error sending message: {e}")
        return False

# Example usage
send_custom_message("What is the capital of France?")

In [ ]:
# Close producer when done
producer.close()
print("Producer closed.")